# Day 2 — Modèles, validation par patient et soumissions

Ce notebook remplace les scripts `archives_py/experiment_day2*.py` (étapes 1 à 7 de `docs/DAY2.md`). Pour chaque étape : une idée, un modèle, un score. Les réponses au workbook sont à la fin.

Les rapports sont déjà sur Skore Hub (workspace `M&C`, projet `bobathon-esilv`) : `PUSH = False` évite de les renvoyer en double.

In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
if not os.path.exists("data"):   # le notebook est dans notebooks_finaux/ : on se place à la racine du repo
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
from sklearn.base import clone
import skore
from skore import evaluate

skore.configuration.show_progress = False  # pas de barres de progression dans le notebook
PUSH = False  # les rapports sont déjà sur Skore Hub : True seulement pour les renvoyer

def push(key, report):
    """Envoie un rapport sur Skore Hub (workspace lu dans .skore) si PUSH vaut True."""
    if not PUSH:
        print(f"(PUSH=False) rapport {key!r} déjà sur le Hub, non renvoyé")
        return
    from skore import Project, login
    cfg = json.load(open(".skore", encoding="utf-8"))
    os.environ["SKORE_HUB_URI"], os.environ["SKORE_HUB_API_KEY"] = cfg["hub_url"], cfg["api_key"]
    login(mode="hub")
    Project(name="bobathon-esilv", mode="hub", workspace=cfg["workspace"]).put(key, report)

In [2]:
def cv_rmse(report, data_source="test"):
    # RMSE de chacun des 5 folds d'un rapport de validation croisée
    return report.metrics.rmse(aggregate=None, data_source=data_source).iloc[0].to_numpy(dtype=float)

def show(name, report):
    # RMSE moyen ± écart entre folds, et écart train/test (surapprentissage)
    test, train = cv_rmse(report), cv_rmse(report, "train")
    print(f"{name:<22} test {test.mean():.3f} ± {test.std(ddof=1):.3f}   train {train.mean():.3f}   écart {test.mean() - train.mean():+.3f}")
    return test

In [3]:
from cv_splits import FEATURE_COLS, load_visits, make_grouped_cv_splits

visits = load_visits()
X_test = pd.read_csv("data/X_test.csv")
X = visits[FEATURE_COLS]   # les 8 colonnes numériques du Day 1
y = visits["target"]
# rmse() d'un rapport simple renvoie un nombre ; cv_rmse() sert pour la validation croisée

## Étape 1 — Ridge contre dummy

**Ridge** est une régression linéaire : prédiction ≈ constante + somme de (poids × colonne). Le paramètre `alpha` réduit les poids pour qu'une colonne bruitée ne domine pas.

Ridge n'accepte pas les valeurs manquantes : on les remplace d'abord par la **médiane** (`SimpleImputer`). Même découpage aléatoire (20 %) que le dummy, pour comparer à armes égales.

In [4]:
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

dummy = DummyRegressor(strategy="mean")
# pipeline : on remplit les trous par la médiane, puis on applique Ridge
ridge = make_pipeline(SimpleImputer(strategy="median"), Ridge(alpha=1.0))

report_dummy = evaluate(dummy, X, y, splitter=0.2)
report_ridge = evaluate(ridge, X, y, splitter=0.2)
rmse_dummy, rmse_ridge = report_dummy.metrics.rmse(), report_ridge.metrics.rmse()
print(f"RMSE dummy {rmse_dummy:.3f} | RMSE ridge {rmse_ridge:.3f} | gain {100 * (1 - rmse_ridge / rmse_dummy):.1f} %")
push("02_ridge", report_ridge)

RMSE dummy 16.478 | RMSE ridge 10.387 | gain 37.0 %
(PUSH=False) rapport '02_ridge' déjà sur le Hub, non renvoyé


Les coefficients de Ridge (attention : les colonnes n'ont pas la même unité) :

In [5]:
fitted = clone(ridge).fit(X, y)   # fitted[-1] = la dernière étape du pipeline (Ridge)
pd.DataFrame({"coef": fitted[-1].coef_, "écart-type de la colonne": X.std()}, index=FEATURE_COLS).round(3).sort_values("coef", key=abs, ascending=False)

,coef,écart-type de la colonne
time_since_intake_on,1.749,1.169
age,1.073,11.536
age_at_diagnosis,-0.957,10.854
off,0.617,16.551
on,0.517,10.365
sexM,-0.180,0.490
time_since_intake_off,-0.157,2.841
ledd,-0.001,218.506


## Étape 2 — Régler `alpha` et soumettre
On essaie plusieurs valeurs d'`alpha` et on garde la meilleure.

In [6]:
rows = []
for alpha in [0.1, 1, 10, 100, 1000]:
    model = make_pipeline(SimpleImputer(strategy="median"), Ridge(alpha=alpha))
    rows.append({"alpha": alpha, "RMSE": evaluate(model, X, y, splitter=0.2).metrics.rmse(),
                 "coef time_since_intake_on": clone(model).fit(X, y)[-1].coef_[FEATURE_COLS.index("time_since_intake_on")]})
pd.DataFrame(rows).round(3)

,alpha,RMSE,coef time_since_intake_on
0,0.1,10.387,1.750
1,1.0,10.387,1.749
2,10.0,10.387,1.749
3,100.0,10.387,1.744
4,1000.0,10.387,1.697


Soumission Kaggle : le pipeline réentraîné sur **toutes** les visites du train, puis prédiction sur `X_test`.

In [7]:
final = clone(ridge).fit(X, y)   # réentraîné sur toutes les visites du train
submission = X_test[["Index"]].copy()
submission["target"] = final.predict(X_test[FEATURE_COLS])
submission.to_csv("submissions/02_ridge.csv", index=False)
submission.shape

(11013, 2)

## Étape 3 — Validation croisée par patient (GroupKFold)

Un découpage aléatoire met des visites **du même patient** dans le train et dans le test : le modèle peut « reconnaître » le patient, c'est une **fuite**. Kaggle, lui, teste sur des patients **nouveaux**.

`GroupKFold` coupe les **patients** en 5 groupes : on entraîne sur 4, on teste sur le 5ᵉ, cinq fois. Les découpages (`cv_splits`) viennent de `cv_splits.py` et sont réutilisés jusqu'à la fin.

In [8]:
cv_splits = make_grouped_cv_splits(visits)   # 5 couples (indices train, indices test)

# vérification : aucun patient ne doit être des deux côtés d'un fold
ids = visits["patient_id"].to_numpy()
print("patients communs train/test par fold :", [len(set(ids[tr]) & set(ids[te])) for tr, te in cv_splits])

report_dummy_gcv = evaluate(dummy, X, y, splitter=cv_splits)   # splitter=cv_splits : validation par patient
ridge = make_pipeline(SimpleImputer(strategy="median"), Ridge(alpha=0.1))   # meilleur alpha de l'étape 2
report_ridge_gcv = evaluate(ridge, X, y, splitter=cv_splits)
show("dummy", report_dummy_gcv)
show("ridge", report_ridge_gcv)
push("03_dummy_gcv", report_dummy_gcv); push("03_ridge_gcv", report_ridge_gcv)

patients communs train/test par fold : [0, 0, 0, 0, 0]
dummy                  test 16.499 ± 0.343   train 16.499   écart +0.000
ridge                  test 10.437 ± 0.197   train 10.424   écart +0.012
(PUSH=False) rapport '03_dummy_gcv' déjà sur le Hub, non renvoyé
(PUSH=False) rapport '03_ridge_gcv' déjà sur le Hub, non renvoyé


## Étape 4 — HistGradientBoosting : le manque comme information

Un **arbre de décision** pose des questions oui/non (`off` manquant ? `age > 62` ?). Le **boosting** enchaîne des centaines de petits arbres, chacun corrigeant les erreurs des précédents.

Avantage clé : HGBR gère les valeurs manquantes **sans les remplacer**. Un `off` manquant devient une information (« examen OFF sauté »), au lieu d'être noyé dans une médiane.

In [9]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

hgbr = HistGradientBoostingRegressor(random_state=0)   # pas d'imputer : HGBR gère les NaN
report_hgbr_gcv = evaluate(hgbr, X, y, splitter=cv_splits)
hgbr_folds = show("hgbr", report_hgbr_gcv)
print("HGBR gagne sur chaque fold :", bool((hgbr_folds < cv_rmse(report_ridge_gcv)).all()))
push("04_hgbr_gcv", report_hgbr_gcv)

# temps moyen d'entraînement et de prédiction sur les 5 folds
for name, r in [("ridge", report_ridge_gcv), ("hgbr", report_hgbr_gcv)]:
    print(f"temps {name}: fit {r.metrics.fit_time(aggregate=None).iloc[0].mean():.3f} s, "
          f"predict {r.metrics.predict_time(aggregate=None).iloc[0].mean():.4f} s")

hgbr                   test 7.434 ± 0.140   train 6.907   écart +0.527
HGBR gagne sur chaque fold : True
(PUSH=False) rapport '04_hgbr_gcv' déjà sur le Hub, non renvoyé
temps ridge: fit 0.017 s, predict 0.0009 s
temps hgbr: fit 0.671 s, predict 0.0110 s


Importance par permutation, mesurée sur un **fold de test** (patients jamais vus) : on mélange une colonne et on regarde de combien le RMSE se dégrade.

In [10]:
tr, te = cv_splits[0]   # fold 0 : entraînement sur tr, mesure sur te (patients nouveaux)
m = clone(hgbr).fit(X.iloc[tr], y.iloc[tr])
perm = permutation_importance(m, X.iloc[te], y.iloc[te], n_repeats=5, random_state=0, scoring="neg_root_mean_squared_error")
pd.Series(perm.importances_mean, index=FEATURE_COLS).sort_values(ascending=False).round(3)

off                      8.366
on                       5.462
age                      2.218
age_at_diagnosis         1.714
time_since_intake_on     0.345
ledd                     0.045
time_since_intake_off    0.004
sexM                    -0.000
dtype: float64

## Étape 5 — skrub `tabular_pipeline` : ajouter les colonnes texte

`gene` et `cohort` sont des chaînes de caractères. `tabular_pipeline("regressor")` choisit un encodage **par colonne** (selon le nombre de valeurs distinctes), puis applique HGBR. On retire les identifiants `Index` et `patient_id`.

In [11]:
from skrub import tabular_pipeline

X_full = visits.drop(columns=["Index", "patient_id", "target"])   # on garde gene et cohort
report_tab_gcv = evaluate(tabular_pipeline("regressor"), X_full, y, splitter=cv_splits)
show("hgbr (étape 4)", report_hgbr_gcv); show("tabular_pipeline", report_tab_gcv)
push("05_tabular_gcv", report_tab_gcv)

# encodeur choisi par le TableVectorizer (1re étape du pipeline)
tv = tabular_pipeline("regressor").fit(X_full, y)[0]
pd.DataFrame({col: {"encodeur": type(tv.transformers_[col]).__name__, "valeurs distinctes": X_full[col].nunique()}
              for col in ["cohort", "gene", "off"]}).T

hgbr (étape 4)         test 7.434 ± 0.140   train 6.907   écart +0.527


tabular_pipeline       test 7.431 ± 0.116   train 6.851   écart +0.580
(PUSH=False) rapport '05_tabular_gcv' déjà sur le Hub, non renvoyé


,encodeur,valeurs distinctes
cohort,ToCategorical,2
gene,ToCategorical,4
off,PassThrough,93


## Étape 6 — skrub DataOps : la validation dans le graphe

Un **DataOp** décrit la recette (entrées, cible, modèle, découpage par patient) en un seul objet : impossible d'oublier `groups`. On suit le code du guide, qui ne retire **que** `target` : `Index` et `patient_id` restent dans les variables, c'est l'objet des questions.

In [12]:
import skrub
from sklearn.model_selection import GroupKFold

# la recette : X = visites sans target, GroupKFold sur patient_id, puis vectorisation + HGBR
data = skrub.var("visits", visits)
X_op = data.drop("target", axis=1).skb.mark_as_X(cv=GroupKFold(n_splits=5), split_kwargs={"groups": data["patient_id"]})
y_op = data["target"].skb.mark_as_y()
pred = X_op.skb.apply(skrub.TableVectorizer()).skb.apply(HistGradientBoostingRegressor(random_state=0), y=y_op)

report_dataops_gcv = evaluate(pred, data={"visits": visits})   # pas de splitter : il est dans le graphe
show("tabular (étape 5)", report_tab_gcv); show("DataOps", report_dataops_gcv)
push("06_dataops_gcv", report_dataops_gcv)

tabular (étape 5)      test 7.431 ± 0.116   train 6.851   écart +0.580


DataOps                test 7.566 ± 0.134   train 6.347   écart +1.220
(PUSH=False) rapport '06_dataops_gcv' déjà sur le Hub, non renvoyé


Les identifiants servent-ils au modèle sur des patients **nouveaux** ? Importance par permutation sur un fold de test :

In [13]:
Xid = visits.drop(columns="target")   # comme X_op : Index et patient_id inclus
m = make_pipeline(skrub.TableVectorizer(), HistGradientBoostingRegressor(random_state=0)).fit(Xid.iloc[tr], y.iloc[tr])
perm = permutation_importance(m, Xid.iloc[te], y.iloc[te], n_repeats=5, random_state=0, scoring="neg_root_mean_squared_error")
pd.Series(perm.importances_mean, index=Xid.columns).sort_values(ascending=False).round(3)

off                      8.081
on                       5.327
age                      1.299
age_at_diagnosis         0.748
time_since_intake_on     0.286
ledd                     0.092
cohort                   0.012
gene                     0.011
sexM                     0.001
time_since_intake_off    0.001
patient_id              -0.006
Index                   -0.006
dtype: float64

## Étape 7 — Modèle final et soumission
Le meilleur modèle du guide est `tabular_pipeline` (étape 5). On le réentraîne sur tout le train et on prédit le test, avec exactement les mêmes colonnes.

In [14]:
model = tabular_pipeline("regressor")
report_final = evaluate(model, X_full, y)   # EstimatorReport : c'est son URL qui va dans la description Kaggle
print(f"RMSE test {report_final.metrics.rmse():.3f} | train {report_final.metrics.rmse(data_source='train'):.3f}")
push("07_tabular", report_final)

final = clone(model).fit(X_full, y)
submission = X_test[["Index"]].copy()
# mêmes colonnes, dans le même ordre qu'au train
submission["target"] = final.predict(X_test.drop(columns=["Index", "patient_id"])[X_full.columns])
submission.to_csv("submissions/07_tabular.csv", index=False)
submission.shape

RMSE test 7.279 | train 6.868
(PUSH=False) rapport '07_tabular' déjà sur le Hub, non renvoyé


(11013, 2)

### Récapitulatif (validation croisée par patient)

In [15]:
pd.DataFrame({name: [cv_rmse(r).mean(), cv_rmse(r).std(ddof=1), cv_rmse(r).mean() - cv_rmse(r, "train").mean()]
              for name, r in [("dummy", report_dummy_gcv), ("ridge", report_ridge_gcv), ("hgbr", report_hgbr_gcv),
                              ("tabular_pipeline", report_tab_gcv), ("DataOps (avec ids)", report_dataops_gcv)]},
             index=["RMSE", "écart entre folds", "écart train/test"]).T.round(3)

,RMSE,écart entre folds,écart train/test
dummy,16.499,0.343,0.000
ridge,10.437,0.197,0.012
hgbr,7.434,0.140,0.527
tabular_pipeline,7.431,0.116,0.580
DataOps (avec ids),7.566,0.134,1.220


---
## Questions du workbook

### Étape 1 — Ridge contre dummy
1. **Gain.** RMSE 10,39 contre 16,48 : Ridge réduit l'erreur de 37 %. Gros gain, mais mesuré sur un découpage aléatoire, donc un peu optimiste.
2. **Coefficients.** Le plus gros en valeur absolue est `time_since_intake_on` (1,75), puis `age` (1,07) et `age_at_diagnosis` (−0,96). `off` (0,62) et `on` (0,52) sont positifs, comme attendu : un score mesuré élevé donne un vrai OFF élevé.
3. **Échelle.** `ledd` varie de centaines de mg (écart-type 219), `age` de quelques années : un coefficient dépend de l'unité, sa taille brute ne dit pas l'importance.

### Étape 2 — Réglage d'`alpha`
1. Le RMSE ne bouge pas (10,39 de 0,1 à 1000) : avec 8 variables et 44 000 lignes, la régularisation a peu d'effet.
2. Les coefficients changent très peu : `time_since_intake_on` passe de 1,75 à 1,70.

### Étape 3 — Validation croisée par patient
1. Dummy : 16,48 → 16,50 ± 0,34. Ridge : 10,39 → 10,44 ± 0,20. Les deux bougent peu : un modèle linéaire à 8 variables ne mémorise presque pas les patients.
2. Pour notre meilleur modèle (`10_env_stack`) : GroupKFold 3,369, découpage aléatoire 3,074, Kaggle public 3,339 et privé 3,192. Kaggle colle à la GroupKFold : c'est elle qu'il faut suivre.

### Étape 4 — HistGradientBoosting
1. HGBR bat Ridge sur les 5 folds (7,43 ± 0,13 contre 10,44 ± 0,18) : le gain de 3 points dépasse largement la variation entre folds.
2. HGBR met environ 0,7 s à s'entraîner contre 0,02 s pour Ridge (≈ 40 fois plus lent), mais reste rapide : le gain vaut le surcoût.
3. HGBR s'appuie surtout sur `off` et `on`, puis l'âge. Le gros coefficient Ridge de `time_since_intake_on` venait de son unité (heures), pas de son utilité.
- Surapprentissage : écart train/test de +0,53 contre +0,01 pour Ridge. HGBR mémorise un peu, sans gravité.

### Étape 5 — `tabular_pipeline`
1. Non : environ 7,43 dans les deux cas (différence de 0,004 au plus), bien moins que l'écart entre folds (± 0,12). `gene` et `cohort` n'apportent rien.
2. Les colonnes numériques passent telles quelles ; `cohort` (2 valeurs) et `gene` (4 valeurs) deviennent des catégories (`ToCategorical`) : peu de valeurs, pas besoin de `StringEncoder`.

### Étape 6 — DataOps
1. Non : environ 7,55 contre 7,43, nettement pire. Deux raisons : `Index` et `patient_id` restent dans les variables, et `TableVectorizer()` seul encode `cohort`/`gene` en one-hot au lieu de catégories.
2. `X_op` a 12 colonnes (les 10 de l'étape 5 + `Index` et `patient_id`). Sur des patients nouveaux leur importance est nulle, mais le modèle s'en sert sur le train (écart train/test qui double) : il mémorise les patients. Il faut les retirer.

### Étape 7 — Bilan
1. Le plus gros gain : dummy → Ridge (−37 %), puis HGBR qui garde les valeurs manquantes (−29 %). Ajouter `gene`/`cohort` n'apporte rien ; garder les identifiants dégrade le score.
2. Les visites les plus mal prédites n'ont souvent pas de `off` (58 % contre 42 %) : sans l'examen OFF, le modèle perd son meilleur signal.
3. La **progression dans le temps** n'est pas traitée : chaque visite est vue seule. Prochaine étape : utiliser les mesures `off`/`on` des **autres visites du même patient** (jamais le `target`, absent du test). C'est l'objet du notebook 03.